Why revenue is not proportional to Profit/Margin

In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv(r"C:\Users\Talib\Desktop\DA - Projects\New Project Resume\Dataset\files v2\Cleaned Data\Fact_Sales.csv")
df

,Row_ID,Order_ID,Date_ID,Order_Date,Ship_Date,Delivery_Duration(Days),Ship_Mode,Customer_ID,Product_ID,Quantity,Unit_Price_BDT,Discount_Pct,Sales_BDT,Cost_BDT,Profit_BDT,Profit_Margin_Pct,Payment_Method,Is_Returned
0,1,ORD-105638,DATE-20230101,01-01-23,02-01-23,1,Standard,CUS-02321,PROD-00001,8,4771.66,0.042,36570.00,34620.64,1949.36,5.33,Cash on Delivery,False
1,2,ORD-102334,DATE-20230101,01-01-23,02-01-23,1,Standard,CUS-01456,PROD-00002,3,11470.75,0.100,30971.03,22946.73,8024.30,25.91,bKash,True
2,3,ORD-100570,DATE-20230101,01-01-23,08-01-23,7,Standard,CUS-00028,PROD-00003,11,58.85,0.077,597.50,548.44,49.06,8.21,Cash on Delivery,False
3,4,ORD-101183,DATE-20230101,01-01-23,02-01-23,1,Standard,CUS-00412,PROD-00004,4,11474.86,0.099,41355.40,38073.88,3281.52,7.93,bKash,False
4,5,ORD-104389,DATE-20230101,01-01-23,03-01-23,2,Standard,CUS-01955,PROD-00005,4,1722.10,0.196,5538.27,5080.03,458.24,8.27,Cash on Delivery,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9085,9086,ORD-103318,DATE-20251230,30-12-25,04-01-26,5,Standard,CUS-00533,PROD-07713,3,49009.23,0.052,139382.25,134636.13,4746.12,3.41,Cash on Delivery,False
9086,9087,ORD-108236,DATE-20251231,31-12-25,02-01-26,2,Same-Day,CUS-00895,PROD-01022,1,4270.98,0.213,3361.26,2882.84,478.42,14.23,bKash,False
9087,9088,ORD-106406,DATE-20251231,31-12-25,05-01-26,5,Standard,CUS-01913,PROD-02484,3,43433.88,0.051,123656.26,119053.06,4603.20,3.72,Cash on Delivery,False
9088,9089,ORD-103140,DATE-20251231,31-12-25,02-01-26,2,Standard,CUS-01687,PROD-07714,11,644.09,0.596,2862.34,-297.56,3159.90,110.40,Credit/Debit Card,False


In [29]:
product = pd.read_csv(r"C:\Users\Talib\Desktop\DA - Projects\New Project Resume\Dataset\files v2\Cleaned Data\Dim_Product.csv")
product

,Product_ID,Category,Sub_Category,Product_Name
0,PROD-00001,Electronics,Smartphone,Smartphone Model-491
1,PROD-00002,Home_Living,Furniture,Furniture Model-360
2,PROD-00003,Grocery_FMCG,Beverages,Beverages Model-461
3,PROD-00004,Home_Living,Bedding,Bedding Model-774
4,PROD-00005,Garments,Kids Wear,Kids Wear Model-303
...,...,...,...,...
7709,PROD-07710,Beauty_Health,Makeup,Makeup Model-747
7710,PROD-07711,Electronics,Headphone,Headphone Model-858
7711,PROD-07712,Grocery_FMCG,Cleaning Supplies,Cleaning Supplies Model-994
7712,PROD-07713,Electronics,Laptop,Laptop Model-966


Is cost increasing faster than revenue

In [52]:
customer = pd.read_csv(r"C:\Users\Talib\Desktop\DA - Projects\New Project Resume\Dataset\files v2\Cleaned Data\Dim_Customer.csv")
customer

,Customer_ID,Segment,Division,District
0,CUS-00001,B2C,Dhaka,Narayanganj
1,CUS-00002,B2C,Mymensingh,Jamalpur
2,CUS-00003,B2C,Khulna,Jessore
3,CUS-00005,B2C,Mymensingh,Mymensingh
4,CUS-00008,B2C,Chittagong,Feni
...,...,...,...,...
1497,CUS-02591,B2C,Chittagong,Chittagong
1498,CUS-02592,B2C,Dhaka,Narayanganj
1499,CUS-02594,B2B,Khulna,Khulna
1500,CUS-02596,B2C,Dhaka,Narayanganj


In [6]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='%d-%m-%y')
yearly_cost = (
    df.groupby(df['Order_Date'].dt.year)
      .agg(
          Revenue=('Sales_BDT', 'sum'),
          Cost=('Cost_BDT', 'sum'),
          Profit=('Profit_BDT', 'sum')
      )
      .reset_index()
)

yearly_cost['Revenue_Growth_Pct'] = (
    yearly_cost['Revenue'].pct_change() * 100
)

yearly_cost['Cost_Growth_Pct'] = (
    yearly_cost['Cost'].pct_change() * 100
)

yearly_cost['Cost_Ratio_Pct'] = (
    yearly_cost['Cost'] /
    yearly_cost['Revenue'] * 100
)

print(yearly_cost.round(2))

   Order_Date      Revenue         Cost      Profit  Revenue_Growth_Pct  \
0        2023  98215745.32  89299910.83  8915834.49                 NaN   
1        2024  80917292.80  73022253.55  7926550.62              -17.61   
2        2025  97097772.71  88213726.05  8884046.66               20.00   

   Cost_Growth_Pct  Cost_Ratio_Pct  
0              NaN           90.92  
1           -18.23           90.24  
2            20.80           90.85  


Is discount affect the margin?

In [11]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='%d-%m-%y')
discount_analysis = (
    df.groupby(df['Order_Date'].dt.year)
      .agg(
          Revenue=('Sales_BDT', 'sum'),
          Profit=('Profit_BDT', 'sum'),
          Avg_Discount=('Discount_Pct', 'mean')
      )
      .reset_index()
)

discount_analysis['Margin_Pct'] = (
    discount_analysis['Profit'] /
    discount_analysis['Revenue'] * 100
)

print(discount_analysis.round(2))


# Correlation
corr = df[['Discount_Pct', 'Profit_Margin_Pct']].corr()

print("\nDiscount vs Margin Correlation")
print(corr.round(3))

   Order_Date      Revenue      Profit  Avg_Discount  Margin_Pct
0        2023  98215745.32  8915834.49          0.11        9.08
1        2024  80917292.80  7926550.62          0.11        9.80
2        2025  97097772.71  8884046.66          0.11        9.15

Discount vs Margin Correlation
                   Discount_Pct  Profit_Margin_Pct
Discount_Pct              1.000              0.497
Profit_Margin_Pct         0.497              1.000


Product Mix Impact

In [51]:
# ১. df-এ থাকা পুরোনো Duplicate (_x, _y) কলামগুলো মুছে ফেলা
df = df.drop(columns=[c for c in df.columns if '_x' in c or '_y' in c], errors='ignore')

# ২. Fact_Sales (df) এবং Dim_Product (product) মার্জ করা
df_product = df.merge(
    product[['Product_ID', 'Category', 'Sub_Category', 'Product_Name']],
    on='Product_ID',
    how='left'
)

# ৩. Category অনুযায়ী Revenue, Profit ও Quantity এগ্রিগেট করা
product_mix = (
    df_product.groupby('Category')
    .agg(
        Revenue=('Sales_BDT', 'sum'),
        Profit=('Profit_BDT', 'sum'),
        Quantity=('Quantity', 'sum')
    )
    .reset_index()
)

# ৪. Revenue Share (%) এবং Profit Margin (%) হিসাব করা
product_mix['Revenue_Share_Pct'] = (
    product_mix['Revenue'] / product_mix['Revenue'].sum() * 100
)

product_mix['Profit_Margin_Pct'] = (
    product_mix['Profit'] / product_mix['Revenue'] * 100
)

# ৫. Revenue অনুযায়ী সাজিয়ে আউটপুট প্রিন্ট করা
print(product_mix.sort_values('Revenue', ascending=False).round(2))

           Category       Revenue       Profit  Quantity  Revenue_Share_Pct  \
2       Electronics  1.944951e+08  14337805.16      6507              70.41   
6       Home_Living  2.800449e+07   3097596.52      4114              10.14   
4          Garments  2.204429e+07   4260883.78     10784               7.98   
5      Grocery_FMCG  1.038494e+07    905825.52     18371               3.76   
3          Footwear  1.019176e+07   1442441.25      3885               3.69   
0     Beauty_Health  7.978225e+06   1342322.37      4823               2.89   
1  Books_Stationery  3.132042e+06    339557.17      3720               1.13   

   Profit_Margin_Pct  
2               7.37  
6              11.06  
4              19.33  
5               8.72  
3              14.15  
0              16.82  
1              10.84  


In [53]:
df_region = df.merge(
    customer[
        ['Customer_ID', 'Segment', 'Division', 'District']
    ],
    on='Customer_ID',
    how='left'
)

region_analysis = (
    df_region.groupby('Division')
    .agg(
        Revenue=('Sales_BDT', 'sum'),
        Profit=('Profit_BDT', 'sum'),
        Quantity=('Quantity', 'sum')
    )
    .reset_index()
)

region_analysis['Margin_Pct'] = (
    region_analysis['Profit'] /
    region_analysis['Revenue'] * 100
)

region_analysis['Revenue_Share_Pct'] = (
    region_analysis['Revenue'] /
    region_analysis['Revenue'].sum() * 100
)

print(
    region_analysis
    .sort_values('Revenue', ascending=False)
    .round(2)
)

     Division       Revenue       Profit  Quantity  Margin_Pct  \
2       Dhaka  1.120431e+08  10448431.80     22948        9.33   
1  Chittagong  6.322784e+07   5586442.71     10014        8.84   
3      Khulna  3.558769e+07   3316220.58      6628        9.32   
7      Sylhet  1.598158e+07   1694653.65      3095       10.60   
5    Rajshahi  1.575808e+07   1732202.39      3450       10.99   
0    Barishal  1.399921e+07   1252607.15      2713        8.95   
4  Mymensingh  1.042082e+07    828465.16      2013        7.95   
6     Rangpur  9.212470e+06    867408.33      1343        9.42   

   Revenue_Share_Pct  
2              40.56  
1              22.89  
3              12.88  
7               5.79  
5               5.70  
0               5.07  
4               3.77  
6               3.34  


Index(['Product_ID', 'Category', 'Sub_Category', 'Product_Name'], dtype='str')
